# ANÁLISIS Y LIMPIEZA DE DATOS ORIGINALES

**Fuente de datos**: https://www.kaggle.com/datasets/ggtejas/tmdb-imdb-merged-movies-dataset

## Preparación del entorno

In [74]:
#%pip install pandas

In [75]:
import pandas as pd

## Carga del dataframe y análisis del estado actual de los datos

In [76]:
ruta = 'raw/TMDB-IMDB-Movies-Dataset.csv'
df = pd.read_csv(ruta)

In [77]:
print('TAMAÑO',df.shape)
print('COLUMNAS')
print(df.columns)
print('TIPOS DE DATOS')
print(df.dtypes.to_string())

TAMAÑO (442717, 29)
COLUMNAS
Index(['id', 'title', 'vote_average', 'vote_count', 'status', 'release_date',
       'revenue', 'runtime', 'adult', 'backdrop_path', 'budget', 'homepage',
       'tconst', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'tagline', 'genres',
       'production_companies', 'production_countries', 'spoken_languages',
       'keywords', 'directors', 'writers', 'averageRating', 'numVotes',
       'cast'],
      dtype='str')
TIPOS DE DATOS
id                        int64
title                       str
vote_average            float64
vote_count                int64
status                      str
release_date                str
revenue                   int64
runtime                   int64
adult                      bool
backdrop_path               str
budget                    int64
homepage                    str
tconst                      str
original_language           str
original_title              str
overview      

In [78]:
print('VALORES FALTANTES')
m = df.isna().sum()
faltantes =pd.DataFrame(
    {'cantidad':m[m.gt(0)],'porcentaje':(m[m.gt(0)]/len(df)*100).round(2)}
    )
faltantes.sort_values(by='porcentaje', ascending=False)


VALORES FALTANTES


,cantidad,porcentaje
homepage,387100,87.44
tagline,348959,78.82
keywords,271242,61.27
backdrop_path,255927,57.81
production_companies,179599,40.57
production_countries,121140,27.36
spoken_languages,109138,24.65
genres,83341,18.82
poster_path,78949,17.83
cast,69539,15.71


La descripción de la película (columna overview) falta en el 10.10% de las películas

Para mi aplicación requiero de la imagen del póster de la película. Si la imagen está en **homepage**, según el reporte de datos faltantes se perderían 87.44% de los datos porque no tienen **homepage**. Más adelante es necesario verificar si ese enlace es el que contiene el póster o si es **poster_path** que tiene menos datos faltantes.

In [79]:
print('DUPLICADOS',df.duplicated().sum())

DUPLICADOS 0


No hay películas duplicadas

In [80]:
print('VISTA DEL PRINCIPIO DEL DATAFRAME')
print(df.head(4).to_string(index=False))

VISTA DEL PRINCIPIO DEL DATAFRAME
    id           title  vote_average  vote_count   status release_date    revenue  runtime  adult                    backdrop_path    budget                                       homepage    tconst original_language  original_title                                                                                                                                                                                                                                                                                                                                                                                                     overview  popularity                      poster_path                                                    tagline                                      genres                                                            production_companies                     production_countries                   spoken_languages                                  

In [81]:
print("VISTA DE LAS PRIMERAS 3 PELÍCULAS DEL DATAFRAME SIN TRUNCAR INFORMACIÓN")
pd.set_option('display.max_colwidth', None)

for indice, fila in df.head(3).iterrows():
    display(fila.to_frame(name=f'Fila {indice}'))

VISTA DE LAS PRIMERAS 3 PELÍCULAS DEL DATAFRAME SIN TRUNCAR INFORMACIÓN


,Fila 0
id,27205
title,Inception
vote_average,8.364
vote_count,34495
status,Released
release_date,2010-07-15
revenue,825532764
runtime,148
adult,False
backdrop_path,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg


,Fila 1
id,157336
title,Interstellar
vote_average,8.417
vote_count,32571
status,Released
release_date,2014-11-05
revenue,701729206
runtime,169
adult,False
backdrop_path,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg


,Fila 2
id,155
title,The Dark Knight
vote_average,8.512
vote_count,30619
status,Released
release_date,2008-07-16
revenue,1004558444
runtime,152
adult,False
backdrop_path,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg


Según la impresión en la celda anterior y la información encontrada en internet:

`backdrop_path` y `poster_path` son rutas que identifican imágenes asociadas a una película en **TMDB**. 
`poster_path` corresponde al **póster vertical** de la película, mientras que `backdrop_path` corresponde a la **imagen horizontal de fondo** (backdrop), generalmente utilizada para encabezados o fondos de una interfaz. 

Estas columnas contienen únicamente la ruta de la imagen, por lo que es necesario agregar la URL base de TMDB para obtener el enlace completo a la imagen.

1. Se agrega la URL base de TMDB delante del valor de poster_path o backdrop_path. Por ejemplo:

https://image.tmdb.org/t/p/original + poster_path

Así, una ruta como /oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg se convierte en un enlace completo: https://image.tmdb.org/t/p/original/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg.

2. Además, se puede indicar el tamaño de la imagen. Por ejemplo, para posters se puede usar w500 y para fondos w1280:

https://image.tmdb.org/t/p/w500 + poster_path
https://image.tmdb.org/t/p/w1280 + backdrop_path

Así se obtiene un enlace completo y se controla la resolución de la imagen según el uso que tendrá en la aplicación.


In [82]:
print('ESTADÍSTICAS DESCRIPTIVAS')
print(df.describe(include='number').to_string())

print('VALORES ÚNICOS EN COLUMNAS DE TEXTO')
print(df.select_dtypes(include='str').nunique().to_string())

ESTADÍSTICAS DESCRIPTIVAS
                 id   vote_average     vote_count       revenue        runtime        budget     popularity  averageRating      numVotes
count  4.427170e+05  442717.000000  442717.000000  4.427170e+05  442717.000000  4.427170e+05  442717.000000  442717.000000  4.427170e+05
mean   5.102150e+05       3.556666      48.045106  1.641487e+06      66.290673  6.436195e+05       2.161301       6.223866  3.235158e+03
std    3.861546e+05       3.158976     508.286438  2.604470e+07      64.232801  7.375499e+06      11.260063       1.314101  3.476341e+04
min    2.000000e+00       0.000000       0.000000  0.000000e+00       0.000000  0.000000e+00       0.000000       1.000000  5.000000e+00
25%    2.139360e+05       0.000000       0.000000  0.000000e+00      15.000000  0.000000e+00       0.600000       5.400000  2.400000e+01
50%    4.260350e+05       4.500000       1.000000  0.000000e+00      80.000000  0.000000e+00       0.843000       6.300000  7.300000e+01
75%    7.307890

## Limpieza de datos

Según la información visualizada anteriormente y el objetivo del uso de los datos defino:
1. Eliminar las películas que hasta la fecha (octubre de 2026) no se han estrenado
2. Usar poster_path como fuente de la imágen de la película y eliminar las películas sin poster_path, un 17.83% de los datos
3. Eliminar el 10.10% de las películas sin descripción en la columna overview
4. Eliminar un aproximado de 60% a 70% de las películas, descartando las que tienen bajo rating (columna averageRating). Esto es para reducir el tamaño del archivo y evitar recomendaciones de películas con baja recepción del público.
5.  Eliminar columnas innecesarias para la app: 
```
       - 'vote_average', se usará en su lugar 'averageRating'
       - 'vote_count', se usará en su lugar 'numVotes'
       - 'status', todas son 'released'
       - 'revenue', 
       - 'backdrop_path', se usará en su lugar 'poster_path'
       - 'budget', 
       - 'homepage',
       - 'tconst', 
       - 'production_companies', 
       - 'production_countries', 
       - 'spoken_languages'
       - 'original_language'
```


### 1. Eliminar las películas que hasta la fecha (octubre de 2026) no se han estrenado

In [83]:
print("Valores únicos de la columna status:")
df['status'].unique()

Valores únicos de la columna status:


<ArrowStringArray>
[       'Released',         'Rumored', 'Post Production',   'In Production',
         'Planned',        'Canceled']
Length: 6, dtype: str

In [84]:
print("TOTAL DE FILAS ANTES DE FILTRAR ", df.shape[0])
df = df[df['status'] == 'Released']
print("TOTAL DE FILAS DESPUÉS DE FILTRAR ", df.shape[0])

print("Valores únicos de la columna status:")
df['status'].unique()

TOTAL DE FILAS ANTES DE FILTRAR  442717
TOTAL DE FILAS DESPUÉS DE FILTRAR  439823
Valores únicos de la columna status:


<ArrowStringArray>
['Released']
Length: 1, dtype: str

### 2. Eliminar las películas sin poster_path

In [85]:
print("TOTAL DE FILAS ANTES DE FILTRAR ", df.shape[0])
df = df[df['poster_path'].notnull()]
print("TOTAL DE FILAS DESPUÉS DE FILTRAR ", df.shape[0])

TOTAL DE FILAS ANTES DE FILTRAR  439823
TOTAL DE FILAS DESPUÉS DE FILTRAR  362162


### 3. Eliminar el 10.10% de las películas sin descripción en la columna overview

In [86]:
print("TOTAL DE FILAS ANTES DE FILTRAR ", df.shape[0])
df = df[df['overview'].notnull()]
print("TOTAL DE FILAS DESPUÉS DE FILTRAR ", df.shape[0])

TOTAL DE FILAS ANTES DE FILTRAR  362162
TOTAL DE FILAS DESPUÉS DE FILTRAR  332936


### 4. Eliminar un aproximado de 50% de las películas
Descartando las que tienen bajo rating (columna averageRating).

In [87]:
print('ESTADÍSTICAS DE LA COLUMNA averageRating')
print(df["averageRating"].describe(include='number').to_string())

ESTADÍSTICAS DE LA COLUMNA averageRating
count    332936.000000
mean          6.204307
std           1.281930
min           1.000000
25%           5.500000
50%           6.300000
75%           7.100000
max          10.000000


In [88]:
print('CANTIDAD DE PELÍCULAS CON averageRating MAYOR A 7')
print((df["averageRating"] >= 7).sum())

CANTIDAD DE PELÍCULAS CON averageRating MAYOR A 7
95711


In [89]:
print("TOTAL DE FILAS ANTES DE FILTRAR ", df.shape[0])
df = df[df['averageRating']>= 7]
print("TOTAL DE FILAS DESPUÉS DE FILTRAR ", df.shape[0])

TOTAL DE FILAS ANTES DE FILTRAR  332936
TOTAL DE FILAS DESPUÉS DE FILTRAR  95711


### 5.  Eliminar columnas innecesarias para la app: 

In [90]:
print("TOTAL DE COLUMNAS ANTES DE FILTRAR ", df.shape[1])
df = df.drop(columns=[
    'vote_average', 
    'vote_count', 
    'status', 
    'revenue', 
    'backdrop_path', 
    'budget', 
    'homepage',
    'tconst', 
    'production_companies', 
    'production_countries', 
    'spoken_languages',
    'original_language'
])

print("TOTAL DE COLUMNAS DESPUÉS DE FILTRAR ", df.shape[1])

print('COLUMNAS')
display(df.columns)


TOTAL DE COLUMNAS ANTES DE FILTRAR  29
TOTAL DE COLUMNAS DESPUÉS DE FILTRAR  17
COLUMNAS


Index(['id', 'title', 'release_date', 'runtime', 'adult', 'original_title',
       'overview', 'popularity', 'poster_path', 'tagline', 'genres',
       'keywords', 'directors', 'writers', 'averageRating', 'numVotes',
       'cast'],
      dtype='str')

## Guardado de datos procesados

In [91]:
df.to_csv("movies.csv", index=False)
#index=False: en to_csv, evita guardar el índice de pandas como una columna extra del archivo.